# FHRPY — False-signal detection on the **GPU** (PyTorch)

The false-signal networks (`FSDop` / `FSScalp`) are bidirectional stacked GRUs.
The bundled NumPy implementation runs the recurrence with a Python time loop;
here we map the **same trained weights** into `torch.nn.GRU` (cuDNN) so the model
runs on the GPU, and we **verify it matches the NumPy reference exactly**.

In [ ]:
# === Setup ===
import sys, pathlib
_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "fhrpy" / "__init__.py").exists()), None)
if _root and str(_root) not in sys.path: sys.path.insert(0, str(_root))
for _m in [k for k in list(sys.modules) if k == "fhrpy" or k.startswith("fhrpy.")]:
    del sys.modules[_m]
import fhrpy, torch
print("fhrpy", fhrpy.__version__, "| torch", torch.__version__,
      "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Build the GPU model and verify it matches NumPy

`FSDopTorch` maps the Keras `reset_after` GRU weights into `torch.nn.GRU`. In
float64 on the CPU the output is identical to the NumPy reference (~1e-14); the
default float32 GPU path agrees to ~1e-3.

In [ ]:
import numpy as np
import fhrpy.datasets as ds
from fhrpy.io import read_fhr
from fhrpy.falsesignal.detect import load_model
from fhrpy.falsesignal.features import build_dop_features
from fhrpy.falsesignal.torch_backend import FSDopTorch

rec = read_fhr(ds.example_path("ctg_example_01"))
ref = load_model("doppler")(build_dop_features(rec.fhr1, rec.mhr))[0]   # NumPy reference

exact = FSDopTorch(device="cpu", dtype=torch.float64).detect(rec.fhr1, rec.mhr)
gpu   = FSDopTorch().detect(rec.fhr1, rec.mhr)                          # default: float32 GPU
print("CPU float64  max|diff| =", float(np.max(np.abs(ref - exact))))   # ~1e-14 (exact)
print("GPU float32  max|diff| =", float(np.max(np.abs(ref - gpu))))     # ~1e-3

## 2. Throughput — GPU vs NumPy CPU

The recurrence (a Python loop in NumPy) is exactly what the GPU accelerates.
Processing several hours of recordings: the GPU is ~40x faster on this machine
(RTX 5070, float32).

In [ ]:
import time
names = ["fs_dopmhr_train0006", "fs_dopmhr_train0010", "fs_dopmhr_train0022", "ctg_example_01", "ctg_example_06"]
recs  = [read_fhr(ds.example_path(n)) for n in names]
feats = [build_dop_features(r.fhr1, r.mhr).T for r in recs]   # (T, 5)
M = load_model("doppler"); T = FSDopTorch()
T.forward(feats[0])  # warm up the GPU

t = time.time(); [M(I.T) for I in feats]; t_cpu = time.time() - t
if torch.cuda.is_available(): torch.cuda.synchronize()
t = time.time(); [T.forward(I) for I in feats]
if torch.cuda.is_available(): torch.cuda.synchronize()
t_gpu = time.time() - t

hours = sum(len(r) / 4 / 3600 for r in recs)
print(f"{len(recs)} records, {hours:.1f} h of signal")
print(f"NumPy CPU : {t_cpu:5.1f} s")
print(f"torch GPU : {t_gpu:5.1f} s   -> speedup x{t_cpu / t_gpu:.0f}")

> **Notes.** The model is the same as `detect_false_signals` (verified above),
> just on the GPU. Very long single recordings (> ~10 h) can exceed a cuDNN GRU
> limit — split them into chunks. The WMFB baseline's heavy stage is a weighted
> median bank (data-dependent, sequential); its Butterworth filter-bank can also
> be batched on the GPU, but the median step stays CPU-friendly — the NumPy CPU
> path already meets the real-time target (see issue #6 / `examples/benchmark.py`).